In [1]:
from pathlib import Path
%load_ext autoreload
%autoreload 2
while not (Path.cwd() / '.git').exists():
    %cd ..

/home/matthew/study/grab-voc-triage


In [2]:
import pandas as pd
import config
import numpy as np
from src.data.transform import encode_labels

In [3]:
df = pd.read_csv("data/preds/openai/grab_reviews__ver2__gpt-4o-mini.csv", index_col = 0)
df[config.CATEGORIES] = df[np.array(config.CATEGORIES) + '_pred']
df.drop(np.array(config.CATEGORIES) + '_pred', axis = 1, inplace = True)
df.index.name = None

In [4]:
for c in config.CATEGORIES:
    print(df[c].value_counts())

DRIVER_OPS
ABSENT    3123
NEG        490
Name: count, dtype: int64
APP_AND_MAPS
ABSENT    2534
NEG       1079
Name: count, dtype: int64
PRICING_AND_BILLING
ABSENT    3013
NEG        600
Name: count, dtype: int64


In [5]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
splitter = MultilabelStratifiedShuffleSplit(n_splits = 1, test_size = 0.2, random_state = config.RANDOM_STATE)
for train_idx, valid_idx in splitter.split(df[['content']], df[config.CATEGORIES]):
    pass


In [6]:
df[config.CATEGORIES].replace({'ABSENT' : 0, 'NEG' : 1}).astype('int64')

,DRIVER_OPS,APP_AND_MAPS,PRICING_AND_BILLING
0,0,0,0
1,0,1,0
2,0,0,0
3,1,0,0
4,0,1,0
...,...,...,...
3668,0,0,0
3669,1,0,0
3670,0,1,0
3671,1,0,0


In [8]:
encode_labels(df[['content'] + config.CATEGORIES]).to_parquet(config.TRAIN_PATH, index = False)